# Caso resuelto: alerta temprana de desercion academica

Este notebook documenta el analisis que alimenta el dashboard. Los datos son **sinteticos**, reproducibles y no representan estudiantes reales.

**Pregunta:** ¿es posible estimar la probabilidad de desercion con informacion academica, de participacion y contexto disponible durante el semestre?


## 1. Preparacion

Ejecute primero `python generar_datos.py` y `python entrenar_modelo.py` para reproducir todos los archivos. La division usa 75% de entrenamiento y 25% de prueba con estratificacion.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

ROOT = Path.cwd()
data = pd.read_csv(ROOT / 'data' / 'datos_estudiantes_ficticios.csv')
resultados = pd.read_csv(ROOT / 'outputs' / 'comparacion_modelos.csv')
pred = pd.read_csv(ROOT / 'outputs' / 'predicciones_prueba.csv')
importancia = pd.read_csv(ROOT / 'outputs' / 'importancia_variables.csv')
data.head()


## 2. Calidad y descripcion de los datos


In [ ]:
print(f'Registros: {len(data):,}')
print(f'Variables: {data.shape[1]}')
print(f'Tasa de desercion: {data.deserta.mean():.1%}')
print(f'Valores faltantes: {data.isna().sum().sum()}')
data.describe(include='all').T


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
data.groupby('programa').deserta.mean().sort_values().plot.barh(ax=axes[0], color='#26A6A1', title='Desercion por programa')
data.groupby('cohorte').deserta.mean().plot(ax=axes[1], marker='o', color='#1976D2', title='Desercion por cohorte')
axes[0].set_xlabel('Proporcion'); axes[1].set_ylabel('Proporcion')
plt.tight_layout();


## 3. Iteraciones del modelo

1. **Linea base:** regresion logistica con asistencia, promedio y materias reprobadas.
2. **Modelo enriquecido:** 19 variables originales, tres variables derivadas, escalamiento, codificacion y regularizacion L1/L2.
3. **Modelo no lineal:** bosque aleatorio con control de profundidad y tamano minimo de hoja.

Los hiperparametros se seleccionaron por ROC-AUC mediante validacion cruzada estratificada de cinco pliegues en entrenamiento. El conjunto de prueba se mantuvo separado para la evaluacion final.


In [ ]:
cols = ['version','regularizacion','roc_auc_entrenamiento','roc_auc_prueba','brecha_auc','accuracy','precision','recall','f1']
resultados[cols].round(3)


In [ ]:
resultados.set_index('version')[['roc_auc_prueba','precision','recall','f1']].plot.bar(figsize=(11,4), color=['#1976D2','#26A6A1','#F4B942','#D95D5D'])
plt.ylim(0,1); plt.ylabel('Puntuacion'); plt.title('Comparacion de modelos en prueba'); plt.xticks(rotation=15); plt.tight_layout();


## 4. Evaluacion del modelo seleccionado

El modelo se elige por ROC-AUC. El umbral de 0,50 no es obligatorio: en un sistema de alerta debe ajustarse a la capacidad institucional y al costo de perder un caso frente al costo de revisar una alerta falsa.


In [ ]:
umbral = 0.50
y_pred = (pred.probabilidad_desercion >= umbral).astype(int)
cm = confusion_matrix(pred.deserta, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Continua','Deserta']).plot(cmap='Blues')
plt.title(f'Matriz de confusion - umbral {umbral:.0%}');


In [ ]:
importancia.head(12).sort_values('importancia').plot.barh(x='variable', y='importancia', legend=False, figsize=(8,5), color='#26A6A1')
plt.title('Importancia por permutacion'); plt.xlabel('Cambio medio en ROC-AUC'); plt.tight_layout();


## 5. Conclusiones

- La incorporacion de variables de compromiso, contexto y caracteristicas derivadas mejora la discriminacion frente a la linea base.
- La regularizacion controla complejidad y facilita un modelo estable e interpretable.
- El ROC-AUC no basta: precision, recall, F1 y matriz de confusion cambian con el umbral.
- Las importancias son predictivas y no demuestran causas.
- Para uso real se requieren validacion temporal, auditoria de sesgos, proteccion de datos y revision humana.

El dashboard `app.py` permite filtrar poblaciones y experimentar con el umbral en tiempo real.
